# MLOps Training 2026/2027 — Task 2
## Notebook 1 — Read & Join the Tables

### Objective
Read the Olist tables from the local PostgreSQL database, inspect their structure, understand the grain of each table, validate keys and duplicates, aggregate one-to-many tables before joining, and create a clean ML table containing exactly one row per order.

### Main artifact
`olist_ml_table.parquet`

### Database
- Database: `olist_db`
- User: `ouahibaahmid`
- Host: `localhost`
- Port: `5432`

### Important rule
The final ML table must have exactly one row per `order_id`.


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sqlalchemy import create_engine, text

## 2. Connect to PostgreSQL

The PostgreSQL user does not require a password, so the connection string does not contain a password.

In [2]:
import os
from sqlalchemy import create_engine

DB_USER = os.environ.get("POSTGRES_USER", "ouahibaahmid")
DB_HOST = os.environ.get("POSTGRES_HOST", "localhost")
DB_PORT = int(os.environ.get("POSTGRES_PORT", "5432"))
DB_NAME = os.environ.get("POSTGRES_DB", "olist_db")

DATABASE_URL = os.environ.get(
    "DATABASE_URL",
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"Database engine created successfully for: {DB_NAME}")

Database engine created successfully for: olist_db


## 3. Test the Database Connection

In [3]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user;")
    )
    database, user = result.fetchone()

print(f"Connected to database: {database}")
print(f"Connected as user: {user}")

Connected to database: olist_db
Connected as user: ouahibaahmid


## 4. Load the Required Olist Tables

For this notebook, we use the tables needed to construct an order-level ML dataset:

- `orders`
- `customers`
- `order_items`
- `order_payments`
- `order_reviews`
- `sellers`
- `geolocation`



In [4]:
tables = [
    "orders",
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "sellers",
    "geolocation"
]

data = {}

for table in tables:
    data[table] = pd.read_sql_table(
        table,
        con=engine,
        schema="public"
    )

# Create DataFrame variables for downstream processing
orders = data["orders"]
customers = data["customers"]
order_items = data["order_items"]
order_payments = data["order_payments"]
order_reviews = data["order_reviews"]
sellers = data["sellers"]
geolocation = data["geolocation"]

print("Loaded tables:")
for name, df in data.items():
    print(f"  {name}: {df.shape}")

Loaded tables:
  orders: (99441, 8)
  customers: (99441, 5)
  order_items: (112650, 7)
  order_payments: (103886, 5)
  order_reviews: (99224, 8)
  sellers: (3095, 4)
  geolocation: (1000163, 5)


In [5]:
for name, df in data.items():
    print(f"{name:20} {df.shape[0]:>10,} rows × {df.shape[1]} columns")

orders                   99,441 rows × 8 columns
customers                99,441 rows × 5 columns
order_items             112,650 rows × 7 columns
order_payments          103,886 rows × 5 columns
order_reviews            99,224 rows × 8 columns
sellers                   3,095 rows × 4 columns
geolocation           1,000,163 rows × 5 columns


## 5. Inspect Table Columns

In [6]:
for name, df in data.items():
    print(f"\n===== {name.upper()} =====")
    print(df.columns.tolist())


===== ORDERS =====
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== CUSTOMERS =====
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== ORDER_ITEMS =====
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== ORDER_PAYMENTS =====
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== ORDER_REVIEWS =====
['review_record_id', 'review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== SELLERS =====
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

===== GEOLOCATION =====
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_stat

## 6. Inspect the First Rows

In [7]:
from IPython.display import display
for name, df in data.items():
    print(f"\n===== {name.upper()} =====")
    display(df.head())



===== ORDERS =====


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26



===== CUSTOMERS =====


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



===== ORDER_ITEMS =====


...

,review_record_id,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,1,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,2,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,3,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,5,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53



===== SELLERS =====


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



===== GEOLOCATION =====


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642952,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


## 7. Understand the Grain of Each Table

| Table | Grain |
|---|---|
| `orders` | One row represents one order |
| `customers` | One row represents one customer |
| `order_items` | One row represents one item within an order |
| `order_payments` | One row represents one payment record |
| `order_reviews` | One row represents one review record |
| `sellers` | One row represents one seller |
| `geolocation` | One row represents one lat/lng observation for a ZIP-code prefix (many rows per prefix) |

The `orders` table is the base table because the prediction problem is defined at the order level.

`order_items`, `order_payments`, and `order_reviews` may contain multiple rows for the same order. Therefore, they must be aggregated by `order_id` before being joined to `orders`.



## 8. Assign Tables to Variables

In [8]:
orders = data["orders"]
customers = data["customers"]
order_items = data["order_items"]
order_payments = data["order_payments"]
order_reviews = data["order_reviews"]

## 9. Check Complete Duplicate Rows

In [9]:
for name, df in data.items():
    duplicate_rows = df.duplicated().sum()
    print(f"{name:20} duplicate complete rows: {duplicate_rows:,}")

orders               duplicate complete rows: 0
customers            duplicate complete rows: 0
order_items          duplicate complete rows: 0
order_payments       duplicate complete rows: 0
order_reviews        duplicate complete rows: 0
sellers              duplicate complete rows: 0
geolocation          duplicate complete rows: 261,863


## 10. Validate Primary and Composite Keys

The expected keys are:
- `orders`: `order_id`
- `customers`: `customer_id`
- `order_items`: (`order_id`, `order_item_id`)
- `order_payments`: (`order_id`, `payment_sequential`)
- `order_reviews`: `review_record_id`

In [10]:
print(
    "Orders unique order_id:",
    orders["order_id"].nunique() == len(orders)
)

print(
    "Customers unique customer_id:",
    customers["customer_id"].nunique() == len(customers)
)

print(
    "Order items unique composite key:",
    ~order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).any()
)

print(
    "Order payments unique composite key:",
    ~order_payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).any()
)

print(
    "Order reviews unique review_record_id:",
    ~order_reviews["review_record_id"].duplicated().any()
)

Orders unique order_id: True
Customers unique customer_id: True
Order items unique composite key: True
Order payments unique composite key: True
Order reviews unique review_record_id: True


## 11. Check Missing Values

Missing values are reported but are not automatically removed or filled. Handling missing values belongs to later data preparation and feature engineering steps.

In [11]:
for name, df in data.items():
    print(f"\n===== {name.upper()} =====")

    missing = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    display(missing[missing > 0])


===== ORDERS =====


order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64


===== CUSTOMERS =====


Series([], dtype: int64)


===== ORDER_ITEMS =====


...

review_comment_title      87656
review_comment_message    58247
dtype: int64


===== SELLERS =====


Series([], dtype: int64)


===== GEOLOCATION =====


Series([], dtype: int64)

## 12. Inspect Order Status

The order status is important for understanding the dataset. We do not create the late/on-time target in this notebook yet.

In [12]:
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 13. Check Order Item Multiplicity

`order_items` contains multiple rows for orders that contain multiple items. This confirms why the table must be aggregated before joining it to `orders`.

In [13]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
)

display(items_per_order.describe())

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

In [14]:
items_per_order.sort_values(
    ascending=False
).head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
c05d6a79e55da72ca780ce90364abed9    12
dtype: int64

## 14. Check Payment Multiplicity

An order can have multiple payment records, so payments also need to be aggregated before joining.

In [15]:
payments_per_order = (
    order_payments
    .groupby("order_id")
    .size()
)

display(payments_per_order.describe())

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

## 15. Check Review Multiplicity

Reviews are also inspected at the order level before aggregation.

In [16]:
reviews_per_order = (
    order_reviews
    .groupby("order_id")
    .size()
)

display(reviews_per_order.describe())

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

## 16. Aggregate Order Items

The item table contains multiple rows per order. We aggregate it to one row per `order_id`.

The resulting features are:
- number of items
- number of unique products
- number of unique sellers
- total item value
- total freight value

In [17]:
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        total_item_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

display(items_agg.head())

,order_id,item_count,unique_products,unique_sellers,total_item_value,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


## 17. Validate the Aggregated Item Table

In [18]:
print(
    "Aggregated item table:",
    len(items_agg),
    "rows"
)

print(
    "Unique orders:",
    items_agg["order_id"].nunique()
)

assert items_agg["order_id"].is_unique

print("✓ Item aggregation validated.")

Aggregated item table: 98666 rows
Unique orders: 98666
✓ Item aggregation validated.


## 18. Aggregate Order Payments

Payments can contain multiple records for an order. We aggregate payment information to one row per order.

In [19]:
payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

display(payments_agg.head())

,order_id,payment_count,total_payment_value,max_payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


## 19. Validate the Aggregated Payment Table

In [20]:
print(
    "Aggregated payment table:",
    len(payments_agg),
    "rows"
)

print(
    "Unique orders:",
    payments_agg["order_id"].nunique()
)

assert payments_agg["order_id"].is_unique

print("✓ Payment aggregation validated.")

Aggregated payment table: 99440 rows
Unique orders: 99440
✓ Payment aggregation validated.


## 20. Aggregate Order Reviews

An order can have review records. We aggregate the number of reviews and the average review score to the order level.

In [21]:
reviews_agg = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_count=("review_record_id", "count"),
        average_review_score=("review_score", "mean")
    )
    .reset_index()
)

display(reviews_agg.head())

,order_id,review_count,average_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,4.0
2,000229ec398224ef6ca0657da4fc703e,1,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0


## 21. Validate the Aggregated Review Table

In [22]:
print(
    "Aggregated review table:",
    len(reviews_agg),
    "rows"
)

print(
    "Unique orders:",
    reviews_agg["order_id"].nunique()
)

assert reviews_agg["order_id"].is_unique

print("✓ Review aggregation validated.")

Aggregated review table: 98673 rows
Unique orders: 98673
✓ Review aggregation validated.


## 22. Prepare Customer Data

The `customers` table has one row per `customer_id`, so it can be safely joined directly to `orders`.

In [23]:
customers_small = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

assert customers_small["customer_id"].is_unique

print("✓ Customer table validated.")

✓ Customer table validated.


## 23. Start the Order-Level ML Table

`orders` is used as the base because it already has one row per order.

In [24]:
ml_table = orders.copy()

print("Rows:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)

assert ml_table["order_id"].is_unique

Rows: 99441
Unique orders: 99441


## 24. Join Customer Information

The relationship is:

`orders.customer_id → customers.customer_id`

Because both sides have one row per customer, we validate the join as one-to-one.

In [25]:
ml_table = ml_table.merge(
    customers_small,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Rows after customer join:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)

assert ml_table["order_id"].is_unique

Rows after customer join: 99441
Unique orders: 99441


## 25. Join Aggregated Order Items

The item data has already been aggregated to one row per order. Therefore, this is now a one-to-one join.

In [26]:
ml_table = ml_table.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows after item join:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)

assert ml_table["order_id"].is_unique

Rows after item join: 99441
Unique orders: 99441


## 26. Join Aggregated Payments

In [27]:
ml_table = ml_table.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows after payment join:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)

assert ml_table["order_id"].is_unique

Rows after payment join: 99441
Unique orders: 99441


## 27. Join Aggregated Reviews

In [28]:
ml_table = ml_table.merge(
    reviews_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows after review join:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)

assert ml_table["order_id"].is_unique

Rows after review join: 99441
Unique orders: 99441


## 27b. Prepare Seller Data

An order can involve multiple sellers (see the unique-seller check above), but
`sellers` has one row per seller. To attach a single seller location to each
order, we pick the **primary seller**: the seller of the highest-value item on
the order (ties broken by the lowest `order_item_id`). This mirrors how a


In [29]:
sellers_small = sellers[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

assert sellers_small["seller_id"].is_unique

primary_seller = (
    order_items
    .sort_values(
        ["order_id", "price", "order_item_id"],
        ascending=[True, False, True]
    )
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "seller_id"]]
)

assert primary_seller["order_id"].is_unique

seller_info_agg = primary_seller.merge(
    sellers_small,
    on="seller_id",
    how="left",
    validate="many_to_one"
).drop(columns=["seller_id"])

assert seller_info_agg["order_id"].is_unique

print("Primary-seller table:", len(seller_info_agg), "rows")


Primary-seller table: 98666 rows


## 27c. Join Seller Information

The primary-seller table has one row per order, so this is a one-to-one join,


In [30]:
ml_table = ml_table.merge(
    seller_info_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Rows after seller join:", len(ml_table))
print(
    "Unique orders:",
    ml_table["order_id"].nunique()
)



Rows after seller join: 99441
Unique orders: 99441


## 27d. Prepare Geolocation Data & Compute Customer–Seller Distance

`geolocation` contains many latitude/longitude observations per ZIP-code
prefix. We aggregate it to one centroid (mean lat/lng) per prefix, then look
up a centroid for the customer's ZIP prefix and the primary seller's ZIP
prefix, and compute the great-circle distance between them with the
haversine formula. Some ZIP prefixes may not appear in `geolocation`; those
orders simply get a missing `distance_km`, which is reported like any other


In [31]:
import numpy as np

geolocation_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        lat=("geolocation_lat", "mean"),
        lng=("geolocation_lng", "mean")
    )
    .reset_index()
)

assert geolocation_agg["geolocation_zip_code_prefix"].is_unique

print("Geolocation centroids:", len(geolocation_agg), "unique ZIP prefixes")


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometers between two lat/lng points."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


ml_table = ml_table.merge(
    geolocation_agg.rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "lat": "customer_lat",
        "lng": "customer_lng"
    }),
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

ml_table = ml_table.merge(
    geolocation_agg.rename(columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "lat": "seller_lat",
        "lng": "seller_lng"
    }),
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

assert ml_table["order_id"].is_unique

ml_table["distance_km"] = haversine_km(
    ml_table["customer_lat"], ml_table["customer_lng"],
    ml_table["seller_lat"], ml_table["seller_lng"]
).round(2)

ml_table = ml_table.drop(
    columns=["customer_lat", "customer_lng", "seller_lat", "seller_lng"]
)

distance_missing_pct = ml_table["distance_km"].isna().mean()

print(
    "distance_km missing:",
    ml_table["distance_km"].isna().sum(),
    f"({distance_missing_pct:.2%})"
)



Geolocation centroids: 19015 unique ZIP prefixes
distance_km missing: 1264 (1.27%)


## 28. Final One-Row-Per-Order Validation

This is the most important validation in Notebook 1. The final ML table must contain exactly one row for every order.

In [32]:
print("Final number of rows:", len(ml_table))
print(
    "Final number of unique orders:",
    ml_table["order_id"].nunique()
)

assert len(ml_table) == ml_table["order_id"].nunique()
assert ml_table["order_id"].is_unique

print("✓ One row per order confirmed.")

Final number of rows: 99441
Final number of unique orders: 99441
✓ One row per order confirmed.


## 29. Inspect the Final ML Table

In [33]:
print("Shape:", ml_table.shape)
display(ml_table.head())

Shape: (99441, 26)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_freight_value,payment_count,total_payment_value,max_payment_installments,review_count,average_review_score,seller_zip_code_prefix,seller_city,seller_state,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,8.72,3.0,38.71,1.0,1.0,4.0,9350.0,maua,SP,18.58
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,22.76,1.0,141.46,1.0,1.0,4.0,31570.0,belo horizonte,SP,851.50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,19.22,1.0,179.12,3.0,1.0,5.0,14840.0,guariba,SP,514.41
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,27.20,1.0,72.20,1.0,1.0,5.0,31842.0,belo horizonte,MG,1822.23
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,8.72,1.0,28.62,1.0,1.0,5.0,8752.0,mogi das cruzes,SP,29.68


In [34]:
ml_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_unique_id             99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  object        
 11  customer_state 

## 30. Check Final Missing Values

Missing values are reported for the next stages. We do not remove or impute them in this notebook because this notebook is focused on reading, joining, and creating the order-level artifact.

In [35]:
missing_summary = (
    ml_table
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[missing_summary > 0]
)

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
distance_km                      1264
item_count                        775
seller_state                      775
seller_city                       775
seller_zip_code_prefix            775
total_freight_value               775
total_item_value                  775
unique_sellers                    775
unique_products                   775
review_count                      768
average_review_score              768
order_approved_at                 160
payment_count                       1
total_payment_value                 1
max_payment_installments            1
dtype: int64

## 31. Numerical Summary

In [36]:
display(ml_table.describe())

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,item_count,unique_products,unique_sellers,total_item_value,total_freight_value,payment_count,total_payment_value,max_payment_installments,review_count,average_review_score,seller_zip_code_prefix,distance_km
count,99441,99281,97658,96476,99441,99441.000000,98666.000000,98666.000000,98666.000000,98666.000000,98666.000000,99440.000000,99440.000000,99440.000000,98673.000000,98673.000000,98666.000000,98177.000000
mean,2017-12-31 08:43:12.776581120,2017-12-31 18:35:24.098800128,2018-01-04 21:49:48.138278656,2018-01-14 12:09:19.035542272,2018-01-24 03:08:37.730111232,35137.474583,1.141731,1.038098,1.013622,137.754076,22.823562,1.044710,160.990267,2.930521,1.005584,4.086793,24632.364766,601.641486
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,1003.000000,1.000000,1.000000,1.000000,0.850000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1001.000000,0.000000
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.249999872,2017-09-25 22:07:22.249999872,2017-10-03 00:00:00,11347.000000,1.000000,1.000000,1.000000,45.900000,13.850000,1.000000,62.010000,1.000000,1.000000,4.000000,6429.000000,185.920000
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,24416.000000,1.000000,1.000000,1.000000,86.900000,17.170000,1.000000,105.290000,2.000000,1.000000,5.000000,13566.000000,433.780000
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.249999872,2018-05-25 00:00:00,58900.000000,1.000000,1.000000,1.000000,149.900000,24.040000,1.000000,176.970000,4.000000,1.000000,5.000000,29156.000000,799.370000
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,99990.000000,21.000000,8.000000,5.000000,13440.000000,1794.960000,29.000000,13664.080000,24.000000,3.000000,5.000000,99730.000000,8677.910000
std,NaN,NaN,NaN,NaN,NaN,29797.938996,0.538452,0.226456,0.122297,210.645145,21.650909,0.381166,221.951257,2.715685,0.075060,1.346274,27700.831009,595.132394


## 32. List Final Features

In [37]:
print("Final columns:")
print()

for i, column in enumerate(ml_table.columns, start=1):
    print(f"{i:02d}. {column}")

Final columns:

01. order_id
02. customer_id
03. order_status
04. order_purchase_timestamp
05. order_approved_at
06. order_delivered_carrier_date
07. order_delivered_customer_date
08. order_estimated_delivery_date
09. customer_unique_id
10. customer_zip_code_prefix
11. customer_city
12. customer_state
13. item_count
14. unique_products
15. unique_sellers
16. total_item_value
17. total_freight_value
18. payment_count
19. total_payment_value
20. max_payment_installments
21. review_count
22. average_review_score
23. seller_zip_code_prefix
24. seller_city
25. seller_state
26. distance_km


## 33. Create Artifact Directory

Artifacts are files produced by one notebook that can be consumed by the next notebook without rerunning the previous notebook.

The main artifact for this notebook will be:

`olist_ml_table.parquet`

In [38]:
from pathlib import Path

# Use project-relative artifact path for full portability
artifact_dir = Path.cwd() / "artifacts" / "notebook_01"
artifact_dir.mkdir(parents=True, exist_ok=True)

print("Artifact directory:", artifact_dir.resolve())

Artifact directory: /Users/ouahibaahmid/training mlops/artifacts/notebook_01


In [39]:
artifact_path = artifact_dir / "olist_ml_table.parquet"

ml_table.to_parquet(
    artifact_path,
    engine="pyarrow",
    index=False
)

print(f"✓ Saved: {artifact_path}")

✓ Saved: /Users/ouahibaahmid/training mlops/artifacts/notebook_01/olist_ml_table.parquet


## 35. Save Data Quality Summary

A small summary is saved alongside the main ML table so that the next steps can inspect the structure and data quality without rerunning this notebook.

In [40]:
data_quality_summary = pd.DataFrame({
    "column": ml_table.columns,
    "dtype": ml_table.dtypes.astype(str).values,
    "missing_count": ml_table.isna().sum().values,
    "unique_count": ml_table.nunique().values
})

quality_path = artifact_dir / "data_quality_summary.csv"

data_quality_summary.to_csv(
    quality_path,
    index=False
)

print(f"✓ Saved: {quality_path}")

✓ Saved: /Users/ouahibaahmid/training mlops/artifacts/notebook_01/data_quality_summary.csv


## 36. Verify the Saved Artifact

The artifact is loaded again to make sure that it can be read successfully and still contains exactly one row per order.

In [41]:
check_artifact = pd.read_parquet(
    artifact_path
)

print("Artifact shape:", check_artifact.shape)

print(
    "Unique orders:",
    check_artifact["order_id"].nunique()
)

assert check_artifact["order_id"].is_unique

print("✓ Artifact loaded successfully.")
print("✓ One row per order confirmed.")

Artifact shape: (99441, 26)
Unique orders: 99441
✓ Artifact loaded successfully.
✓ One row per order confirmed.


# Final Summary

Notebook 1 successfully completed the following steps:

1. Connected to the local PostgreSQL `olist_db` database.
2. Loaded the required Olist tables.
3. Inspected table dimensions and columns.
4. Identified the grain of each table.
5. Checked duplicates and key uniqueness.
6. Inspected missing values.
7. Investigated the one-to-many relationships.
8. Aggregated `order_items` to the order level.
9. Aggregated `order_payments` to the order level.
10. Aggregated `order_reviews` to the order level.
11. Chose a primary seller per order and joined seller state/city/ZIP.
12. Aggregated `geolocation` to one centroid per ZIP prefix and computed
    `distance_km` between the customer and the primary seller.
13. Joined customer information to orders.
14. Joined the aggregated order-level tables.
15. Validated that the final dataset contains exactly one row per order.
16. Saved the final ML table as a Parquet artifact.
17. Saved a data-quality summary.

### Main artifact

`artifacts/notebook_01/olist_ml_table.parquet`

### Next step

The next notebook can read this artifact without querying the original tables again.
